# Day 22 — Implement EDA API Endpoints

**Goal:** Implement schema-aware EDA endpoints inside the existing `django_api/ipl_api` app without replacing your existing player APIs.

## Endpoints
- `GET /api/eda/summary/`
- `GET /api/eda/batting/`
- `GET /api/eda/bowling/`
- `GET /api/eda/season/`
- `GET /api/eda/venue/`
- `GET /api/eda/opposition/`
- `GET /api/eda/recent-form/?player_name=...`

## Safety and schema
The uploaded `views.py` confirms batting/bowling staging table names and basic player statistics. Run this notebook from your repository root after Day 22 schema inspection has run. The notebook reads actual MySQL columns via Django, builds SQL only from identifiers found in that schema, and **does not overwrite existing `views.py` or `urls.py`**. It writes a separate module and a URL patch for you to review and merge.

Run the notebook cells in order. If the schema report says fields are missing, the corresponding endpoint returns a clear HTTP 400 rather than querying a guessed column.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, csv, shutil
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    candidates = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if candidates:
        REPO_ROOT = candidates[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
VIEWS_PATH = APP_DIR / "views.py"
URLS_PATH = APP_DIR / "urls.py"
REPORT_DIR = REPO_ROOT / "reports" / "day22"
DOCS_DIR = REPO_ROOT / "docs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

assert MANAGE_PY.exists(), f"Could not find {MANAGE_PY}; run from project root."
assert VIEWS_PATH.exists(), f"Could not find {VIEWS_PATH}."
assert URLS_PATH.exists(), f"Could not find {URLS_PATH}."

print("Project root:", REPO_ROOT)
print("Django root:", DJANGO_ROOT)
print("Existing views and URLs will be preserved.")


Project root: D:\Git\Personal\ipl-player-performance
Django root: D:\Git\Personal\ipl-player-performance\django_api
Existing views and URLs will be preserved.


In [2]:
# Inspect the real staging schema through Django's configured MySQL connection.
schema_script = r'''
import os, json
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django
django.setup()
from django.db import connection
tables = ["stg_ipl_batting_player_match", "stg_ipl_bowling_player_match", "stg_ipl_match_metadata"]
result = {}
with connection.cursor() as cursor:
    for table in tables:
        cursor.execute(
            "SELECT COLUMN_NAME, DATA_TYPE FROM information_schema.columns "
            "WHERE table_schema = DATABASE() AND table_name = %s ORDER BY ORDINAL_POSITION",
            [table],
        )
        result[table] = [{"column": r[0], "data_type": r[1]} for r in cursor.fetchall()]
print(json.dumps(result))
'''
env = os.environ.copy()
env["PYTHONPATH"] = str(DJANGO_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["DJANGO_SETTINGS_MODULE"] = "config.settings"
schema_proc = subprocess.run([sys.executable, "-c", schema_script], cwd=str(DJANGO_ROOT), env=env, text=True, capture_output=True)
print("Schema command return code:", schema_proc.returncode)
print(schema_proc.stdout)
if schema_proc.stderr:
    print(schema_proc.stderr)
if schema_proc.returncode != 0:
    raise RuntimeError("Schema inspection failed; fix the displayed Django/MySQL error before continuing.")
schema = json.loads(schema_proc.stdout.strip().splitlines()[-1])
(REPORT_DIR / "mysql_staging_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")


Schema command return code: 0
{"stg_ipl_batting_player_match": [{"column": "match_id", "data_type": "bigint"}, {"column": "date", "data_type": "text"}, {"column": "season", "data_type": "text"}, {"column": "venue", "data_type": "text"}, {"column": "city", "data_type": "text"}, {"column": "team_1", "data_type": "text"}, {"column": "team_2", "data_type": "text"}, {"column": "match_type", "data_type": "double"}, {"column": "team", "data_type": "text"}, {"column": "opposition", "data_type": "text"}, {"column": "player_name", "data_type": "text"}, {"column": "player_id", "data_type": "text"}, {"column": "player_name_norm", "data_type": "text"}, {"column": "team_norm", "data_type": "text"}, {"column": "opposition_norm", "data_type": "text"}, {"column": "runs", "data_type": "bigint"}, {"column": "balls_faced", "data_type": "bigint"}, {"column": "fours", "data_type": "bigint"}, {"column": "sixes", "data_type": "bigint"}, {"column": "batting_position", "data_type": "bigint"}, {"column": "was_di

3996

In [3]:
# Generate an isolated EDA module with SQL using only verified identifiers.
# The endpoint methods validate optional filters and return JSON via DRF.
schema_cols = {table: {x["column"] for x in cols} for table, cols in schema.items()}
bat_table = "stg_ipl_batting_player_match"
bowl_table = "stg_ipl_bowling_player_match"
meta_table = "stg_ipl_match_metadata"

# Discover optional metadata columns using conservative name candidates.
def choose(table, candidates):
    cols = schema_cols.get(table, set())
    for name in candidates:
        if name in cols:
            return name
    return None

cfg = {
    "bat_table": bat_table,
    "bowl_table": bowl_table,
    "meta_table": meta_table,
    "player": choose(bat_table, ["player_name"]),
    "runs": choose(bat_table, ["runs"]),
    "balls": choose(bat_table, ["balls_faced"]),
    "fours": choose(bat_table, ["fours"]),
    "sixes": choose(bat_table, ["sixes"]),
    "bat_match": choose(bat_table, ["match_id"]),
    "wickets": choose(bowl_table, ["wickets"]),
    "runs_conceded": choose(bowl_table, ["runs_conceded"]),
    "overs": choose(bowl_table, ["overs"]),
    "bowl_match": choose(bowl_table, ["match_id"]),
    "date": choose(meta_table, ["date", "match_date", "start_date"]),
    "season": choose(meta_table, ["season", "season_year", "year"]),
    "venue": choose(meta_table, ["venue", "stadium"]),
    "team": choose(meta_table, ["team", "team_name"]),
    "opposition": choose(meta_table, ["opposition", "opponent", "opponent_team"]),
    "meta_match": choose(meta_table, ["match_id", "id"]),
}
(REPORT_DIR / "eda_column_mapping.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print(json.dumps(cfg, indent=2))


{
  "bat_table": "stg_ipl_batting_player_match",
  "bowl_table": "stg_ipl_bowling_player_match",
  "meta_table": "stg_ipl_match_metadata",
  "player": "player_name",
  "runs": "runs",
  "balls": "balls_faced",
  "fours": "fours",
  "sixes": "sixes",
  "bat_match": "match_id",
  "wickets": "wickets",
  "runs_conceded": "runs_conceded",
  "overs": "overs",
  "bowl_match": "match_id",
  "date": "date",
  "season": "season",
  "venue": "venue",
  "team": null,
  "opposition": null,
  "meta_match": "match_id"
}


In [4]:
# Create a separate ipl_api/eda_views.py file; do not overwrite the existing views.py.
# SQL identifiers are quoted and only selected from actual schema inspection.
module = r'''from django.db import connection
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

# Generated from Day 22 MySQL information_schema inspection.
CFG = __CFG__

def qi(name):
    if not name or not str(name).replace("_", "").isalnum():
        raise ValueError("Invalid SQL identifier")
    return "`" + str(name).replace("`", "``") + "`"

def table_count(table):
    with connection.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM " + qi(table))
        return int(cursor.fetchone()[0])

def rows(sql, params=None):
    with connection.cursor() as cursor:
        cursor.execute(sql, params or [])
        names = [col[0] for col in cursor.description] if cursor.description else []
        return [dict(zip(names, row)) for row in cursor.fetchall()]

def unavailable(field):
    return Response({"detail": "EDA endpoint requires a confirmed schema column.", "missing_column": field},
                    status=status.HTTP_400_BAD_REQUEST)

class EDASummaryAPIView(APIView):
    def get(self, request):
        result = {"batting_records": table_count(CFG["bat_table"]),
                  "bowling_records": table_count(CFG["bowl_table"])}
        if CFG["meta_table"] in CFG:
            try:
                result["match_metadata_records"] = table_count(CFG["meta_table"])
            except Exception:
                result["match_metadata_records"] = None
        if CFG["runs"]:
            result["total_runs"] = rows("SELECT COALESCE(SUM(" + qi(CFG["runs"]) + "),0) AS total_runs FROM " + qi(CFG["bat_table"]))[0]["total_runs"]
        if CFG["wickets"]:
            result["total_wickets"] = rows("SELECT COALESCE(SUM(" + qi(CFG["wickets"]) + "),0) AS total_wickets FROM " + qi(CFG["bowl_table"]))[0]["total_wickets"]
        return Response(result)

class EDABattingAPIView(APIView):
    def get(self, request):
        c = CFG
        required = ["player", "runs"]
        missing = [x for x in required if not c.get(x)]
        if missing:
            return unavailable(", ".join(missing))
        select = [qi(c["player"]) + " AS player_name", "COUNT(*) AS innings", "SUM(" + qi(c["runs"]) + ") AS runs"]
        if c["balls"]: select.append("SUM(" + qi(c["balls"]) + ") AS balls_faced")
        if c["fours"]: select.append("SUM(" + qi(c["fours"]) + ") AS fours")
        if c["sixes"]: select.append("SUM(" + qi(c["sixes"]) + ") AS sixes")
        sql = "SELECT " + ", ".join(select) + " FROM " + qi(c["bat_table"]) + " WHERE " + qi(c["player"]) + " IS NOT NULL GROUP BY " + qi(c["player"]) + " ORDER BY runs DESC"
        return Response(rows(sql))

class EDABowlingAPIView(APIView):
    def get(self, request):
        c = CFG
        if not c["player"] or not c["wickets"] or not c["runs_conceded"]:
            return unavailable("player_name, wickets, runs_conceded")
        select = [qi(c["player"]) + " AS player_name", "COUNT(*) AS bowling_records",
                  "SUM(" + qi(c["wickets"]) + ") AS wickets",
                  "SUM(" + qi(c["runs_conceded"]) + ") AS runs_conceded"]
        if c["overs"]: select.append("SUM(" + qi(c["overs"]) + ") AS overs")
        sql = "SELECT " + ", ".join(select) + " FROM " + qi(c["bowl_table"]) + " WHERE " + qi(c["player"]) + " IS NOT NULL GROUP BY " + qi(c["player"]) + " ORDER BY wickets DESC, runs_conceded ASC"
        return Response(rows(sql))

class EDASeasonAPIView(APIView):
    def get(self, request):
        c = CFG
        if not c["season"] or not c["meta_match"]:
            return unavailable("season and match_id in match metadata")
        sql = "SELECT " + qi(c["season"]) + " AS season, COUNT(*) AS matches FROM " + qi(c["meta_table"]) + " GROUP BY " + qi(c["season"]) + " ORDER BY " + qi(c["season"])
        return Response(rows(sql))

class EDAVenueAPIView(APIView):
    def get(self, request):
        c = CFG
        if not c["venue"]:
            return unavailable("venue in match metadata")
        sql = "SELECT " + qi(c["venue"]) + " AS venue, COUNT(*) AS matches FROM " + qi(c["meta_table"]) + " WHERE " + qi(c["venue"]) + " IS NOT NULL GROUP BY " + qi(c["venue"]) + " ORDER BY matches DESC"
        return Response(rows(sql))

class EDAOppositionAPIView(APIView):
    def get(self, request):
        c = CFG
        if not c["opposition"]:
            return unavailable("opposition in match metadata")
        sql = "SELECT " + qi(c["opposition"]) + " AS opposition, COUNT(*) AS matches FROM " + qi(c["meta_table"]) + " WHERE " + qi(c["opposition"]) + " IS NOT NULL GROUP BY " + qi(c["opposition"]) + " ORDER BY matches DESC"
        return Response(rows(sql))

class EDARecentFormAPIView(APIView):
    def get(self, request):
        c = CFG
        player_name = (request.query_params.get("player_name") or "").strip()
        if not player_name:
            return Response({"detail": "player_name query parameter is required."}, status=status.HTTP_400_BAD_REQUEST)
        if not c["player"] or not c["bat_match"]:
            return unavailable("player_name and match_id in batting table")
        # Stable sequence uses match_id unless a confirmed date column can be joined.
        sql = "SELECT * FROM " + qi(c["bat_table"]) + " WHERE " + qi(c["player"]) + " = %s ORDER BY " + qi(c["bat_match"]) + " DESC LIMIT 5"
        return Response(rows(sql, [player_name]))
'''
module = module.replace("__CFG__", repr(cfg))
module_path = APP_DIR / "eda_views.py"
if module_path.exists():
    backup = module_path.with_suffix(".py.day22_backup")
    shutil.copy2(module_path, backup)
    print("Backed up previous EDA module to", backup)
module_path.write_text(module, encoding="utf-8")
compile(module, str(module_path), "exec")
print("Created:", module_path)


Backed up previous EDA module to D:\Git\Personal\ipl-player-performance\django_api\ipl_api\eda_views.py.day22_backup
Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\eda_views.py


In [5]:
# Generate a URL patch for manual merge; preserve current urlpatterns and imports.
url_patch = '''# Day 22 EDA URL additions — merge into django_api/ipl_api/urls.py
from .eda_views import (
    EDASummaryAPIView, EDABattingAPIView, EDABowlingAPIView,
    EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView,
    EDARecentFormAPIView,
)

# Add these entries to urlpatterns:
path("eda/summary/", EDASummaryAPIView.as_view(), name="eda-summary"),
path("eda/batting/", EDABattingAPIView.as_view(), name="eda-batting"),
path("eda/bowling/", EDABowlingAPIView.as_view(), name="eda-bowling"),
path("eda/season/", EDASeasonAPIView.as_view(), name="eda-season"),
path("eda/venue/", EDAVenueAPIView.as_view(), name="eda-venue"),
path("eda/opposition/", EDAOppositionAPIView.as_view(), name="eda-opposition"),
path("eda/recent-form/", EDARecentFormAPIView.as_view(), name="eda-recent-form"),
'''
patch_path = APP_DIR / "day22_eda_urls_patch.txt"
patch_path.write_text(url_patch, encoding="utf-8")
print("Created URL patch:", patch_path)
print("IMPORTANT: merge it into existing urls.py; do not replace the file.")


Created URL patch: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\day22_eda_urls_patch.txt
IMPORTANT: merge it into existing urls.py; do not replace the file.


In [6]:
# Run system checks and existing Day 21 regression tests.
def run_cmd(args):
    p = subprocess.run(args, cwd=str(DJANGO_ROOT), text=True, capture_output=True)
    print("$", " ".join(map(str, args)))
    print("Return code:", p.returncode)
    print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

check = run_cmd([sys.executable, str(MANAGE_PY), "check"])
tests = run_cmd([sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"])


$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py check
Return code: 0
System check identified no issues (0 silenced).

$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py test ipl_api.tests.test_pagination_utils_day21 --verbosity 1
Return code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK



In [7]:
# Write Day 22 implementation report with observed check/test outcomes.
doc = [
    "# Day 22 — EDA API Implementation",
    "",
    "- Generated: " + datetime.now().isoformat(timespec="seconds"),
    "- EDA module: `django_api/ipl_api/eda_views.py`",
    "- URL patch: `django_api/ipl_api/day22_eda_urls_patch.txt`",
    "- Django check return code: `" + str(check.returncode) + "`",
    "- Day 21 regression test return code: `" + str(tests.returncode) + "`",
    "",
    "## Endpoints in the generated module",
    "",
    "- `/api/eda/summary/`",
    "- `/api/eda/batting/`",
    "- `/api/eda/bowling/`",
    "- `/api/eda/season/`",
    "- `/api/eda/venue/`",
    "- `/api/eda/opposition/`",
    "- `/api/eda/recent-form/?player_name=...`",
    "",
    "## Important integration steps",
    "",
    "1. Review `reports/day22/eda_column_mapping.json` and verify candidate mappings against the actual schema.",
    "2. Merge `day22_eda_urls_patch.txt` into existing `ipl_api/urls.py`, preserving existing routes and imports.",
    "3. Run `python manage.py check` and the Day 21 tests again.",
    "4. Start the server and request all seven endpoints; verify JSON responses and HTTP 400 for missing required query parameters.",
    "5. Season, venue, and opposition endpoints currently aggregate match metadata fields when available. They are not yet player-by-opponent performance joins.",
    "6. Recent form currently orders by match_id descending. If match_id is not chronological, change it to use a verified match date/order column before claiming chronological recent form.",
    "",
    "## Run status",
    "",
    "This notebook generated a separate module and URL patch; it intentionally did not overwrite the existing views.py or urls.py. Runtime success is determined by the outputs above and by endpoint smoke tests.",
]
(DOCS_DIR / "day22_eda_api_implementation.md").write_text("\n".join(doc) + "\n", encoding="utf-8")
print("Documentation saved:", DOCS_DIR / "day22_eda_api_implementation.md")


Documentation saved: D:\Git\Personal\ipl-player-performance\docs\day22_eda_api_implementation.md


## Completion checklist

- [ ] Schema inspected; review `reports/day22/eda_column_mapping.json`.
- [ ] Generated `eda_views.py` reviewed for correct field mapping.
- [ ] URL patch merged into existing `ipl_api/urls.py`.
- [ ] `python manage.py check` passes after URL merge.
- [ ] Day 21 regression tests pass.
- [ ] Start server and smoke-test all seven EDA routes.
- [ ] Confirm recent-form order is chronological before marking it complete.

**Do not move to Day 23 until the EDA routes are registered and smoke-tested, unless you explicitly choose to defer testing.**